##### Librerías

In [ ]:
# Librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Pipeline y modelo
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

# Explicabilidad
import shap
import lime
import lime.lime_tabular

# Utilidades del proyecto
from data_utils import *


##### Carga y preprocesamiento del dataset

In [ ]:
X, y = load_and_clean_compas(
    './dataset/compas-scores-two-years.csv'
)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Preprocesador
preprocessor = obtener_preprocesador()

##### Modelo XGBoost

In [ ]:
# Modelo XGBoost seleccionado

xgb_model = XGBClassifier(
    colsample_bytree=1.0,
    learning_rate=0.05,
    max_depth=2,
    n_estimators=75,
    subsample=0.2,
    random_state=42
)

##### Construcción del pipeline


In [ ]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", xgb_model)
])

pipeline.fit(X_train, y_train)

##### Explicabilidad global mediante SHAP


In [ ]:
# Preparación SHAP

model = pipeline.named_steps["classifier"]
preprocessor = pipeline.named_steps["preprocessor"]

X_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_transformed = pd.DataFrame(
    X_transformed,
    columns=feature_names,
    index=X_test.index
)

explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(X_transformed)


In [ ]:
# SHAP summary plot

shap.summary_plot(
    shap_values,
    X_transformed
)


##### Explicabilidad local mediante SHAP


In [ ]:
# Individuo a analizar
idx = 0

# Clase real y predicción
pred = model.predict(X_transformed.iloc[[idx]])[0]
prob = model.predict_proba(X_transformed.iloc[[idx]])[0][1]
real = y_test.iloc[idx]

print(f"Clase real: {real}")
print(f"Predicción del modelo: {pred}")
print(f"Probabilidad de reincidencia: {prob:.3f}")

In [ ]:
# Waterfall plot SHAP

shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[idx],
        base_values=explainer.expected_value,
        data=X_transformed.iloc[idx],
        feature_names=feature_names
    )
)


##### Explicabilidad local mediante LIME


In [ ]:
# Crear explainer LIME

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_transformed),
    feature_names=feature_names,
    class_names=['No Recid', 'Recid'],
    mode='classification'
)


In [ ]:
# Explicación local LIME

lime_exp = lime_explainer.explain_instance(
    X_transformed.iloc[idx].values,
    model.predict_proba,
    num_features=10
)

lime_exp.as_pyplot_figure()

plt.title(f"LIME Local Explanation - Individuo {idx}")
plt.show()


##### SHAP con dependencias entre variables


In [ ]:
# Dependence plot - priors_count

shap.dependence_plot(
    "num__priors_count",
    shap_values,
    X_transformed,
    feature_names=feature_names
)


In [ ]:
# Dependence plot con interacción racial

shap.dependence_plot(
    "num__priors_count",
    shap_values,
    X_transformed,
    feature_names=feature_names,
    interaction_index="cat__race_African-American"
)


In [ ]:
# Dependence plot con interacción de edad

shap.dependence_plot(
    "num__priors_count",
    shap_values,
    X_transformed,
    feature_names=feature_names,
    interaction_index="num__age"
)